In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from konlpy.tag import Mecab
from scipy import sparse
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 설정 및 데이터 로드
# ---------------------------------------------------------
mecab = Mecab()
TARGET_DIRS = ["data/대법원_판례_txt화", "data/법률_정제_완료_txt화"]
OUTPUT_FILE = "legal_stopword_analysis_final.csv"

def get_corpus_list(target_dirs):
    corpus = []
    files = []
    for d in target_dirs:
        files.extend(glob.glob(os.path.join(d, "**", "*.txt"), recursive=True))
    
    print(f"총 {len(files)}개의 문서 로딩 및 전처리 시작...")
    
    for file_path in tqdm(files):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            # 인코딩 오류 시 cp949 시도 등 예외처리는 생략 (필요 시 추가)
        except:
            continue
            
        # 명사(N), 용언(V)만 추출 (조사/어미 제외)
        tokens = []
        try:
            for word, pos in mecab.pos(content):
                if pos.startswith('N') or pos.startswith('V'):
                    tokens.append(word)
        except:
            continue
            
        if tokens:
            corpus.append(" ".join(tokens))
            
    return corpus

# 실행
print(">>> 데이터 로딩 중...")
corpus_data = get_corpus_list(TARGET_DIRS)
print(f"전처리 완료된 문서 수: {len(corpus_data)}")

if len(corpus_data) == 0:
    print("[Error] 처리할 문서가 없습니다. 경로를 확인해주세요.")
    exit()

# ---------------------------------------------------------
# 2. DTM 생성 (Sparse Matrix)
# ---------------------------------------------------------
print(">>> DTM(Document-Term Matrix) 생성 중...")
# min_df=5: 5개 미만 문서에 등장한 희귀 단어는 분석에서 제외 (노이즈)
vectorizer = CountVectorizer(min_df=10) 
X = vectorizer.fit_transform(corpus_data)
feature_names = np.array(vectorizer.get_feature_names_out())
print(f"DTM 크기: {X.shape} (문서 수 x 단어 수)")

# ---------------------------------------------------------
# 3. Metric 1: TF-IDF (가중치)
# ---------------------------------------------------------
print(">>> 지표 1: TF-IDF 계산 중...")
tfidf_transformer = TfidfTransformer()
X_tfidf = tfidf_transformer.fit_transform(X)
# 평균 TF-IDF (낮을수록 불용어)
avg_tfidf = np.array(X_tfidf.mean(axis=0)).flatten()

# ---------------------------------------------------------
# 4. Metric 2: Entropy (무작위성)
# ---------------------------------------------------------
print(">>> 지표 2: Entropy 계산 중...")
# P(d|t) 계산
total_term_freq = np.array(X.sum(axis=0)).flatten()
inv_freq = 1.0 / total_term_freq
D_inv = sparse.diags(inv_freq)
P_dt = X @ D_inv  # Column Normalize

# Entropy: -Sum(P * log P)
# log(0) 방지를 위해 data 속성만 건드림
P_dt.data = P_dt.data * np.log(P_dt.data)
entropy_scores = -1 * np.array(P_dt.sum(axis=0)).flatten()

# ---------------------------------------------------------
# 5. Metric 3: MP-VP (평균 확률 & 분산)
# ---------------------------------------------------------
print(">>> 지표 3: MP-VP 계산 중...")
# P(t|d) = 문서 내 단어 빈도 / 문서 전체 단어 수
doc_lengths = np.array(X.sum(axis=1)).flatten()
doc_lengths[doc_lengths == 0] = 1 # div by zero 방지
D_inv_doc = sparse.diags(1.0 / doc_lengths)
P_td = D_inv_doc @ X  # Row Normalize

# MP (Mean Probability)
mp_scores = np.array(P_td.mean(axis=0)).flatten()

# VP (Variance of Probability) = E[X^2] - (E[X])^2
# 메모리 절약을 위해 Sparse 상태에서 제곱 계산
P_td_sq = P_td.copy()
P_td_sq.data **= 2
mean_sq = np.array(P_td_sq.mean(axis=0)).flatten()
vp_scores = mean_sq - (mp_scores ** 2)

# ---------------------------------------------------------
# 6. 결과 통합 및 정규화
# ---------------------------------------------------------
print(">>> 결과 정리 및 저장 중...")
df = pd.DataFrame({
    'word': feature_names,
    'count': total_term_freq,
    'tfidf': avg_tfidf,
    'entropy': entropy_scores,
    'mp': mp_scores,
    'vp': vp_scores
})

# Min-Max Normalization (0~1)
for col in ['tfidf', 'entropy', 'mp', 'vp']:
    df[f'norm_{col}'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# 불용어 점수 (Stopword Score) 계산
# 1. TF-IDF: 낮을수록 불용어 -> (1 - norm)
df['score_tfidf'] = 1 - df['norm_tfidf']
# 2. Entropy: 높을수록 불용어 -> (norm)
df['score_entropy'] = df['norm_entropy']
# 3. MP-VP: MP 높고 VP 낮을수록 불용어 -> norm_mp + (1 - norm_vp)
df['score_mpvp'] = df['norm_mp'] + (1 - df['norm_vp'])

df = df.round(4)

# 저장
df.sort_values('count', ascending=False).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"완료! 파일 저장됨: {OUTPUT_FILE}")

# ---------------------------------------------------------
# 7. 상위 후보 미리보기
# ---------------------------------------------------------
top_n = 20
print(f"\n[각 지표별 Top {top_n} 불용어 후보]")
print("-" * 60)
print(f"{'Rank':<5} | {'TF-IDF (Low Value)':<20} | {'Entropy (High Value)':<20} | {'MP-VP (High Mean/Low Var)':<20}")
print("-" * 60)

top_tfidf = df.sort_values('score_tfidf', ascending=False).head(top_n)['word'].tolist()
top_entropy = df.sort_values('score_entropy', ascending=False).head(top_n)['word'].tolist()
top_mpvp = df.sort_values('score_mpvp', ascending=False).head(top_n)['word'].tolist()

for i in range(top_n):
    print(f"{i+1:<5} | {top_tfidf[i]:<20} | {top_entropy[i]:<20} | {top_mpvp[i]:<20}")

>>> 데이터 로딩 중...
총 7162개의 문서 로딩 및 전처리 시작...


100%|██████████| 7162/7162 [00:34<00:00, 205.45it/s]


전처리 완료된 문서 수: 7162
>>> DTM(Document-Term Matrix) 생성 중...
DTM 크기: (7162, 9807) (문서 수 x 단어 수)
>>> 지표 1: TF-IDF 계산 중...
>>> 지표 2: Entropy 계산 중...
>>> 지표 3: MP-VP 계산 중...
>>> 결과 정리 및 저장 중...
완료! 파일 저장됨: legal_stopword_analysis_final.csv

[각 지표별 Top 20 불용어 후보]
------------------------------------------------------------
Rank  | TF-IDF (Low Value)   | Entropy (High Value) | MP-VP (High Mean/Low Var)
------------------------------------------------------------
1     | 疫學                   | 일치                   | 원심                  
2     | 區分                   | 주심                   | 판결                  
3     | 續行                   | 주문                   | 사건                  
4     | 더한다                  | 대법관                  | 한다                  
5     | 溝渠                   | 이유                   | 관한                  
6     | 끝낼                   | 원심                   | 판단                  
7     | 規程                   | 관여                   | 경우                  
8     | 減輕       